# LoRA fine-tune ESM2-650M on D3 — Google Colab

Run `scripts/12_lora_finetune_d3.py` on a free Colab T4 (≈30–45 min) instead of MPS Mac (≈2–3 hr).

## Before you start

1. **Set the runtime to a GPU**: `Runtime → Change runtime type → Hardware accelerator → T4 GPU` (Free tier is enough; A100 is ~3× faster on Colab Pro+).
2. **Upload your cache files to Google Drive once.**  In your Drive create a folder (default: `MyDrive/uapp_cache/`) and upload these two files from your Mac:
    - `cache/t2837_metadata.csv`  (≈ 1 MB, has `wtAA`, `mutAA`, `rel_rsa`, `pdb_code`, `split`, `mut_idx`, `sequence`, `ddG`, …)
    - `cache/t2837_bio_features_650m.pt`  (small, output of `scripts/06_build_bio_features.py`)

    The 650M cache file is *not* needed — LoRA re-runs the backbone every step.

3. Run the cells below top to bottom.  Final outputs are mirrored to `MyDrive/uapp_cache/outputs_lora_d3_650m/` so you can keep them between sessions.

## 1.  GPU sanity check + install deps

In [ ]:
import torch, sys
print(f"Python:   {sys.version.split()[0]}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError(
        "\n⚠️  No GPU detected.  In Colab go to Runtime → Change runtime type → T4 GPU, "
        "then re-run this cell."
    )

# Colab usually has transformers; peft may need installing.
!pip install -q -U peft transformers

## 2.  Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive containing t2837_metadata.csv and t2837_bio_features_650m.pt.
# Change this if you uploaded them somewhere else.
DRIVE_DIR = '/content/drive/MyDrive/uapp_cache'

import os
if not os.path.isdir(DRIVE_DIR):
    raise FileNotFoundError(
        f'{DRIVE_DIR} not found.  Create it in Drive and upload '
        '`t2837_metadata.csv` and `t2837_bio_features_650m.pt` first.'
    )
print('Drive contents:')
for f in sorted(os.listdir(DRIVE_DIR)):
    p = os.path.join(DRIVE_DIR, f)
    sz = os.path.getsize(p) / 1e6 if os.path.isfile(p) else 0
    print(f'  {f}  ({sz:.2f} MB)')

## 3.  Clone the repo + copy cache locally

Working out of `/content/uapp`.  We copy the small cache files from Drive to local SSD so the script's I/O is fast.

In [ ]:
%cd /content
![ -d uapp ] || git clone https://github.com/RoselindSi/uapp.git
%cd uapp

# If you need a feature branch (script 12 not yet on main), uncomment:
# !git checkout claude/dreamy-curran-08f646

!git pull --ff-only

import shutil, os
os.makedirs('cache', exist_ok=True)
for fn in ['t2837_metadata.csv', 't2837_bio_features_650m.pt']:
    src = os.path.join(DRIVE_DIR, fn)
    dst = os.path.join('cache', fn)
    if not os.path.exists(src):
        raise FileNotFoundError(
            f'Missing {src} — please upload it to Drive first '
            f'(see the markdown at the top of this notebook).'
        )
    shutil.copy(src, dst)
    print(f'✓ {fn}  ({os.path.getsize(dst)/1e6:.2f} MB)')

## 4.  Run the LoRA fine-tune

Hyperparameters are sized for a free T4 (16 GB VRAM).  On A100 you can bump `--batch-size 32` and roughly halve walltime.

**Compute estimate**: ~1–2 min/epoch on T4, ~25–40 min total for 20 epochs.

In [ ]:
!python scripts/12_lora_finetune_d3.py \
    --metadata-csv cache/t2837_metadata.csv \
    --bio-feats    cache/t2837_bio_features_650m.pt \
    --out          outputs/lora_d3_650m \
    --device       cuda \
    --batch-size   16 \
    --max-epochs   20 \
    --patience     5 \
    --lr           5e-4

## 5.  Inspect the results

In [ ]:
import json
from pathlib import Path

OUT = Path('outputs/lora_d3_650m')
metrics = json.loads((OUT / 'test_metrics.json').read_text())
log     = json.loads((OUT / 'training_log.json').read_text())

print('=' * 78)
print(f"LoRA fine-tune of ESM2-650M on D3 (RSA + chemistry)")
print('=' * 78)
print(f"Trained {len(log)} epochs.  Best val_loss = "
      f"{min(e['val_loss'] for e in log):.4f}")

print(f"\nTest metrics (n = {metrics['n']}):")
for k, v in metrics.items():
    if k == 'n': continue
    if isinstance(v, float):
        print(f"  {k:<12} = {v:+.4f}")
    else:
        print(f"  {k:<12} = {v}")

print("\nCompare against the frozen-backbone D3 baseline (script 09 fixed-split):")
print("  Frozen 650M D3:  RMSE 1.50   NLL 1.89   ICE 0.069   Spearman 0.333")
print('=' * 78)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

epochs = [e['epoch'] for e in log]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, [e['train_loss'] for e in log], 'o-', label='train', color='#0D7377')
ax.plot(epochs, [e['val_loss']   for e in log], 's-', label='val',   color='#E8913A')
ax.set_xlabel('epoch'); ax.set_ylabel('Student-t NLL')
ax.set_title('LoRA D3-650M training curves')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6.  Save outputs back to Drive

This mirrors `outputs/lora_d3_650m/` to your Drive so you can pull the LoRA adapter, head weights, and metrics down to your Mac.  Total ~50 MB.

In [ ]:
import shutil, os
DST = os.path.join(DRIVE_DIR, 'outputs_lora_d3_650m')
shutil.copytree('outputs/lora_d3_650m', DST, dirs_exist_ok=True)
for root, _, files in os.walk(DST):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.relpath(p, DRIVE_DIR):<55}  {os.path.getsize(p)/1e6:6.2f} MB')
print(f"\n✓ Mirrored to: {DST}")

## 7.  (Optional) Re-build bio features on Colab from scratch

If you only uploaded `t2837_metadata.csv` and *not* the bio-features file, run this cell to build it on Colab using `scripts/06_build_bio_features.py`.  This takes a few seconds.

You also need the embedding cache file for this to work — script 06 validates row counts against it.  If the embedding cache lives in Drive too:

```python
shutil.copy(os.path.join(DRIVE_DIR, 't2837_embeddings_v2_650m.pt'), 'cache/')
```

Otherwise, build embeddings on Colab too with `scripts/01_cache_embeddings_esm_v2.py --esm-model facebook/esm2_t33_650M_UR50D --device cuda` (~10 min on T4).

In [ ]:
# Only run this if you didn't upload t2837_bio_features_650m.pt to Drive
# !python scripts/06_build_bio_features.py \
#     --metadata-csv cache/t2837_metadata.csv \
#     --embeddings   cache/t2837_embeddings_v2_650m.pt \
#     --rsa-col      rel_rsa \
#     --out          cache/t2837_bio_features_650m.pt

## 8.  (Optional) Re-run the analysis pipeline on Colab

Now that you're on a GPU, you can also redo the multi-seed and K-fold CV runs much faster:

```bash
# Multi-seed (script 09) — ~3 min on T4
!python scripts/09_multiseed_experiment_d.py \
    --embeddings cache/t2837_embeddings_v2_650m.pt \
    --bio-feats  cache/t2837_bio_features_650m.pt \
    --out        outputs/multiseed_650m \
    --device cuda --seeds 0 1 2 3 4 5 6 7

# K-fold CV (script 11) — ~5 min on T4 with K=5 folds × 5 seeds = 25 paired obs
!python scripts/11_kfold_cv_track_d.py \
    --embeddings cache/t2837_embeddings_v2_650m.pt \
    --bio-feats  cache/t2837_bio_features_650m.pt \
    --out        outputs/cv_650m_more_seeds \
    --device cuda --folds 5 --seeds 0 1 2 3 4
```

On a GPU, the bottleneck shifts from compute to test-set noise.  More seeds **directly** tighten your statistical confidence intervals.